# Imagination study: completion, reconstruction and assembly

Run `run_imagination_study.sh`, then set `RUN_GROUP` below and Run All. This notebook embeds image bytes so downloaded executed notebooks retain the evidence. Distances use input-normalized units, not centimetres. Ground truth is used only for evaluation, never prior selection. Smoke runs test wiring only.


In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, HTML
# Run from the repository root or pt_reg. No notebook-time dependency installation.
for candidate in (Path.cwd(), Path.cwd() / "pt_reg", Path("/home/kpandey/satellite/pt_reg")):
    if (candidate / "generative_assembly").is_dir():
        sys.path.insert(0, str(candidate)); break
from generative_assembly.imagination_report import collect, export, discover, gallery
from generative_assembly import data, geometry as g
GA_ROOT = Path(os.environ.get("GA_ROOT", "/home/kpandey/generative_assembly"))
# Set this to the path printed by the runner, or to a downloaded run group.
RUN_GROUP = Path(os.environ.get("RUN_GROUP", str(GA_ROOT / "runs" / "SET_RUN_GROUP_HERE")))
DATASET = os.environ.get("DATASET")  # Optional relocated dataset.json override.
assert discover(RUN_GROUP), f"No study.json found under {RUN_GROUP}; set RUN_GROUP."
rows = export(RUN_GROUP, DATASET)
df = pd.DataFrame(rows)
assert not df.empty, "No results found"
display(df.groupby(["profile", "stage", "status"], dropna=False).size().rename("jobs").to_frame())
display(df[df["status"] != "complete"][["profile", "stage", "arm", "error"]])
print("Portable full metrics:", RUN_GROUP / "analysis" / "metrics.csv")
print("Smoke jobs:", int(df["smoke"].fillna(False).sum()))


## E1: did the model complete the fragment?

Compare `legacy_cleanup → clean` to isolate cleanup, `clean → completion` for the prompt, and `completion → exterior` for mask protection. Compare SD15 with SD15-depth within each profile for conditioning. Invalid outputs are excluded from E2; foreground growth alone is not proof of completion. Missing-region metrics use a sampled point-splat silhouette proxy.


In [ ]:
e1 = df[(df.stage == "E1") & (df.status == "complete")].copy()
keys = ["profile", "model", "input_type"]
metrics = [c for c in ["observed_mask_retention", "growth_relative_to_input", "significant_components",
    "crop_border_fraction", "cleanup_removed_foreground_fraction", "silhouette_iou",
    "missing_region_precision", "missing_region_recall"] if c in e1]
if not e1.empty:
    display(e1.groupby(keys)[metrics].mean().round(4))
    if "valid_foreground" in e1:
        display(e1.groupby(keys)["valid_foreground"].agg(["count", "mean"]).rename(columns={"mean": "valid_fraction"}))
CASE_ID = df.case_id.iloc[0]  # Change to any case ID below.
print("Available cases:", sorted(df.case_id.unique()))
display(HTML(gallery(rows, CASE_ID, max_images=1000)))


## E0: verify the inputs and edit masks

White mask pixels are editable; black pixels are protected. Inspect fracture boundaries and the black unknown regions of depth controls.


In [ ]:
from generative_assembly.imagination_report import input_gallery
display(HTML(input_gallery(RUN_GROUP, CASE_ID)))


## E2: valid aligned shapes and rejection rates

Rejected templates are counted but never compared in the ground-truth coordinate frame. The oracle measures the entire render → InstantMesh → input-only alignment bridge. Compare case means, not the best ground-truth-selected seed. Missing or rejected outputs must remain visible alongside accuracy.


In [ ]:
e2 = df[(df.stage == "E2") & (df.status == "complete")].copy()
if not e2.empty:
    display(e2.groupby(["profile", "model", "input_type"], dropna=False).agg(
        reconstructions=("job_id", "count"), rejected=("template_rejected", "sum"),
        aligned=("valid_alignment", "sum")))
    available = [c for c in ["chamfer", "precision_002", "recall_002", "f1_002", "heldout_error"] if c in e2]
    if available:
        # Each case gets equal weight even if rejection counts differ.
        case_means = e2[e2.valid_alignment.fillna(False)].groupby(
            ["profile", "model", "input_type", "case_id"], dropna=False)[available].mean()
        display(case_means.groupby(["profile", "model", "input_type"], dropna=False).mean().round(4))
        display(e2[e2.case_id == CASE_ID][["profile", "arm", "template_rejected", "rejection_reason"] + available].sort_values(["profile"] + (["chamfer"] if "chamfer" in e2 else []), na_position="last"))

    if "chamfer" in e2:
        measured = e2[e2.valid_alignment.fillna(False)].dropna(subset=["chamfer"])
        case_means = measured.groupby(["profile", "model", "input_type", "case_id"], dropna=False)["chamfer"].mean().reset_index()
        raw_means = case_means[case_means.model == "raw"].drop(columns="model").rename(columns={"chamfer": "raw_chamfer"})
        paired = case_means[~case_means.model.isin(["raw", "true_image"])].merge(
            raw_means, on=["profile", "input_type", "case_id"], how="inner")
        if not paired.empty:
            paired["improvement_over_raw"] = paired.raw_chamfer - paired.chamfer
            display(paired.groupby(["profile", "model", "input_type"]).agg(
                paired_cases=("case_id", "nunique"), mean_improvement=("improvement_over_raw", "mean")))
            print("Positive improvement means lower Chamfer than the matched raw control. Check rejection rates alongside it.")


## Three diverse priors per model/input

Selection uses held-out observed-fragment fit and normalized shape diversity. Ground-truth scores do not choose priors. Fewer than three priors means the run did not produce three valid, distinct shapes; duplicates are never used to pad the result.


In [ ]:
prior_rows = []
for run in discover(RUN_GROUP):
    for path in sorted((run / "priors").glob("*.json")):
        for arm, value in json.loads(path.read_text()).items():
            prior_rows.append(dict(profile=run.name, case_id=path.stem, arm=arm, **value))
priors = pd.DataFrame(prior_rows)
if not priors.empty:
    display(priors[priors.case_id == CASE_ID])
else:
    print("No prior selection artifacts: run E2 with the corrected pipeline.")


## Inspect selected 3D shapes

Choose one profile and case. Reference fragments are transformed into the anchor coordinate frame before plotting. Only selected, valid aligned priors appear. Plotly is optional; the numerical report and embedded images do not require it.


In [ ]:
PROFILE = "exterior"
try:
    import plotly.graph_objects as go
    run = next((r for r in discover(RUN_GROUP) if r.name == PROFILE), None)
    if run is None:
        print("Profile unavailable:", PROFILE)
    else:
        cfg = json.loads((run / "study.json").read_text())["identity"]["config"]
        ds = Path(DATASET) if DATASET else Path(json.loads((run / "experiment.json").read_text())["dataset"])
        fig = go.Figure()
        if ds.exists():
            rec = next(c for c in json.loads(ds.read_text())["cases"] if c["id"] == CASE_ID)
            case = data.load_case(ds, rec, cfg)
            ref = data.reference(ds, case)
            if ref and "complete" in ref:
                p = ref["complete"]
                fig.add_trace(go.Scatter3d(x=p[:,0], y=p[:,1], z=p[:,2], mode="markers",
                    marker=dict(size=2, opacity=.35), name="Complete reference"))
                for i, points in enumerate(case["points"]):
                    p = g.apply(points, ref["poses"][i])
                    fig.add_trace(go.Scatter3d(x=p[:,0], y=p[:,1], z=p[:,2], mode="markers",
                        marker=dict(size=2), name=f"GT-posed fragment {i}"))
        selected_path = run / "priors" / f"{CASE_ID}.json"
        if selected_path.exists():
            for arm, entry in json.loads(selected_path.read_text()).items():
                for rank, jid in enumerate(entry["job_ids"]):
                    with np.load(run / "jobs" / jid / "shape.npz", allow_pickle=False) as f:
                        p = f["aligned"]
                    fig.add_trace(go.Scatter3d(x=p[:,0], y=p[:,1], z=p[:,2], mode="markers",
                        marker=dict(size=1.5, opacity=.5), name=f"{arm} prior {rank+1}",
                        visible="legendonly"))
        fig.update_layout(scene=dict(aspectmode="data"), height=650, title=f"{PROFILE}: {CASE_ID}")
        fig.show()
except ImportError:
    print("Plotly unavailable; inspect metrics and shape.npz artifacts instead.")


## Optional assembly comparison

Run with `STAGES="E0 E1 E2 E4"` to compare each selected prior against B0 (no prior), B4 (true geometry oracle), and B6 (true-image reconstruction). Individual hypotheses remain separate arms; evaluation does not choose one using ground truth.


In [ ]:
e4 = df[(df.stage == "E4") & (df.status == "complete")].copy()
columns = [c for c in ["success", "max_part_chamfer"] if c in e4]
if not e4.empty and columns:
    display(e4.groupby(["profile", "arm"])[columns].mean().round(4))
else:
    print("E4 not run. Establish useful E1/E2 priors before the assembly comparison.")


## What to return for the next iteration

Download this executed notebook, `analysis/metrics.csv`, `analysis/metrics.json`, and run logs. Keep `study.json`, `experiment.json`, job `result.json`, `backend.json`, `alignment.json`, selected `shape.npz`, and `priors/` for diagnosis. If a profile improves missing-region recall but its E2 accuracy stays poor, investigate the 3D bridge/alignment; if raw images look complete but cleaned outputs lose foreground, cleanup is responsible; if the oracle is poor, resolve reconstruction/alignment before scaling image experiments.
